In [ ]:
#Install pandas last version
%pip install -U pandas

#instal word2number module
%pip install word2number

In [216]:
#Import libraries
import pandas as pd
import numpy as np
from word2number import w2n #to transform numbers from word to number 'two' -> 2
import kagglehub as kg #To import dataset from kaggle

In [217]:
##Load the csv file from kagggle

path = kg.dataset_download("kandeelai22/messy-e-commerce-sales-dataset")

df = pd.read_csv(path + '/messy_ecommerce_sales_data.csv') #save the dataset as a dataframe

Using Colab cache for faster access to the 'messy-e-commerce-sales-dataset' dataset.


In [218]:
#Print the first five rows
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [219]:
#Review the dataframe information
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1    Customer_Name  103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5    Category       95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 16.1 KB
None


In [220]:
#Review the columns name
print(df.columns)
#---Some columns name have spaces

#Remove spaces from columns name
df.columns = df.columns.str.replace(' ', '')
print(df.columns)
print(df.info())
#Column names fixed

Index(['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product',
       ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total'],
      dtype='str')
Index(['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category',
       'Quantity', 'Price', 'Payment_Method', 'Status', 'Total'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1   Customer_Name   103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5   Category        95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null    

In [221]:
#Count Null values
print(df.isna().sum())

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64


In [222]:
#===Analysing duplicates====
print(df.duplicated().sum())
print(df[df.duplicated()])

1
      ID Customer_Name   Order_ID Order_Date     Product    Category Quantity  \
102  146  Customer_146  ORD-32755   7/9/2025  Basketball  electronic        2   

      Price Payment_Method      Status    Total  
102  705.42  Bank Transfer  Processing  1410.84  


In [223]:
#Drop dupicates
df = df.drop_duplicates()
print(df.duplicated().sum())#verify duplicates

0


In [224]:
#=========CLEANING Order_Date COLUMN ==========
#Create a function to transform different formats of date into datetime
def parse_date(x):
  try:
    #parse numeric formart date
    return pd.to_datetime(x, format = '%m/%d/%Y')
  except:
    try:
      #string format date
      return pd.to_datetime(x, format = '%b %d %Y')
    except:
      return pd.NaT #if not match the previous formats

#Apply the function in the column order_date
df['Order_Date'] = df['Order_Date'].apply(parse_date)
print(df['Order_Date'].isnull().sum()) #verify if there is null values

#There is a null value, so we can drop it because is just a row
df = df.dropna(subset = ['Order_Date'])
print(df['Order_Date'].isnull().sum())#verify again that there isn't any value after removing nulls


1
0


In [225]:
#====CLEANING Category COLUMN====
print(df['Category'].value_counts()) #to verify if there is any duplicate categories due to typo

#there are some categories that repets many times due to use of upper, lower case or singular
#->1: Transform the upper case to lower case, then capitalize
df['Category'] = df['Category'].str.lower().str.capitalize()


#-> 2: Plurize all the singular categories
df['Category'] = df['Category'].replace('Electronic', 'Electronics')
print(df['Category'].value_counts())

#Now there are just 5 categories

Category
Books          22
Home           20
Sports         16
Clothing       14
Electronics    11
electronic      3
ELECTRONICS     3
electronics     3
sports          1
Name: count, dtype: int64
Category
Books          22
Home           20
Electronics    20
Sports         17
Clothing       14
Name: count, dtype: int64


In [226]:
#Missing values in category column
print(df[df['Category'].isna()])

#Missing values can be filled infering from Product column
#Create a map
mapping = {
    'Biography': 'Books',
    'Headphones': 'Electronics',
    'Laptop': 'Electronics',
    'Smartphone': 'Electronics',
    'Vacuum': 'Electronics',
    'Shoes': 'Clothing',
    'Jeans': 'Clothing',
    'Basketball': 'Sports'
}

#Apply the map function
df['Category'] = df['Category'].fillna(df['Product'].map(mapping))
print(df.info())

     ID Customer_Name   Order_ID Order_Date     Product Category Quantity  \
33  133  Customer_133  ORD-68182 2024-12-05   Biography      NaN        5   
36  136  Customer_136  ORD-20985 2025-06-12  Headphones      NaN        1   
80  180  Customer_180  ORD-86629 2025-03-26      Laptop      NaN        2   
81  181  Customer_181  ORD-54481 2024-12-29  Smartphone      NaN        1   
82  182  Customer_182  ORD-79672 2025-08-03       Shoes      NaN        4   
84  184  Customer_184  ORD-67799 2025-02-15       Jeans      NaN        4   
93  193  Customer_193  ORD-42475 2025-06-06  Basketball      NaN      NaN   
98  198  Customer_198  ORD-14608 2025-07-27      Vacuum      NaN        2   

     Price    Payment_Method      Status    Total  
33  343.24       Credit Card     Shipped  1716.20  
36  696.71       Credit Card   Delivered   696.71  
80  418.38       Credit Card  Processing   836.76  
81  266.48  Cash on Delivery   Cancelled   266.48  
82  538.35            PayPal  Processing  2153

In [227]:
#Transforming the Category column into category
df['Category'] = df['Category'].astype('category')
print(df.info())

<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 non-null    int64         
 1   Customer_Name   101 non-null    str           
 2   Order_ID        101 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         101 non-null    str           
 5   Category        101 non-null    category      
 6   Quantity        96 non-null     str           
 7   Price           96 non-null     str           
 8   Payment_Method  101 non-null    str           
 9   Status          101 non-null    str           
 10  Total           88 non-null     float64       
dtypes: category(1), datetime64[us](1), float64(1), int64(1), str(7)
memory usage: 14.4 KB
None


In [228]:
#=========ClEANING Product COLUMN ========
print(df['Product'].value_counts())

#There is uncapitalized case, shoes and Shoes are the same product
#Capitalize
df['Product'] = df['Product'].str.capitalize()#transform shoes to Shoes
print(df['Product'].value_counts())
print(df.info())

Product
Blender          8
Shoes            8
Tennis Racket    7
Basketball       7
Comics           7
Lamp             7
Science          6
Smartphone       5
Biography        5
Fiction          5
Microwave        5
Yoga Mat         5
Vacuum           5
Laptop           4
T-shirt          3
Smartwatch       3
Headphones       3
Jeans            3
Football         2
Jacket           2
shoes            1
Name: count, dtype: int64
Product
Shoes            9
Blender          8
Tennis racket    7
Basketball       7
Comics           7
Lamp             7
Science          6
Smartphone       5
Biography        5
Fiction          5
Microwave        5
Yoga mat         5
Vacuum           5
Laptop           4
T-shirt          3
Smartwatch       3
Headphones       3
Jeans            3
Football         2
Jacket           2
Name: count, dtype: int64
<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------ 

In [229]:
print(df.groupby(['Category', 'Product'])['ID'].count())
#There are some products that apear in different categories
#1: build maping from data
mode_map = df.groupby('Product')['Category'].agg(lambda x: x.mode()[0])
print(mode_map)

#overwrite the category column to recategorize the products filled wrong
df['Category'] = df['Product'].map(mode_map)

print(df.groupby(['Category', 'Product'])['ID'].count())

Category     Product      
Books        Biography        5
             Comics           7
             Fiction          5
             Science          6
Clothing     Jacket           2
             Jeans            3
             Shoes            9
             T-shirt          2
Electronics  Basketball       1
             Blender          1
             Headphones       3
             Lamp             1
             Laptop           4
             Microwave        1
             Smartphone       5
             Smartwatch       3
             T-shirt          1
             Vacuum           2
             Yoga mat         2
Home         Blender          7
             Lamp             6
             Microwave        4
             Vacuum           3
Sports       Basketball       6
             Football         2
             Tennis racket    7
             Yoga mat         3
Name: ID, dtype: int64
Product
Basketball            Sports
Biography              Books
Blender             

In [230]:
#====CLEANING Quantity COlUMN=========
print(df[df['Quantity'].isna()])

#there are 5 missing values
#fill the NaN value with 1, assuming that customers bought just 1 item from each product
df['Quantity'] = df['Quantity'].fillna(1)

#Transform the column into int data type
df['Quantity'] = df['Quantity'].astype('int')

#there are negative values, but the quantity must be positive values
#transform the negative values into positive
df['Quantity'] = df['Quantity'].abs()
print(df['Quantity'].value_counts())
# Now The column Quanitity was fixed

     ID Customer_Name   Order_ID Order_Date        Product Category Quantity  \
6   106  Customer_106  ORD-25885 2025-02-02        Blender     Home      NaN   
64  164  Customer_164  ORD-23010 2025-02-09  Tennis racket   Sports      NaN   
67  167  Customer_167  ORD-30329 2025-08-07     Basketball   Sports      NaN   
93  193  Customer_193  ORD-42475 2025-06-06     Basketball   Sports      NaN   
96  196  Customer_196  ORD-78384 2024-12-23         Vacuum     Home      NaN   

     Price    Payment_Method      Status  Total  
6   978.63     Bank Transfer  Processing    NaN  
64  587.64       Credit Card    Returned    NaN  
67  593.93  Cash on Delivery  Processing    NaN  
93  522.02            PayPal     Shipped    NaN  
96     abd            PayPal   Delivered    NaN  
Quantity
5    26
1    25
2    21
4    16
3    13
Name: count, dtype: int64


In [231]:
#======CLEANING Price COLUMN=====
print(df[df['Price'].isna()])
#there are missing values, also number as string
#Define a function to tourn string number into number
def str_to_number(x):
  try:
    if isinstance(x, str):
      return w2n.word_to_num(x)
    else:
      return x
  except:
    return x #keep the orinal number
#apply de function
df['Price'] = df['Price'].apply(str_to_number)

#Replace $ by ''
df['Price'] = df['Price'].str.replace('$', '')
#Replace the abd by NaN
df['Price'] = df['Price'].replace('abd', np.nan)

#Turning the Price column into float data type
df['Price'] = df['Price'].astype('float')
print(df.info())

#Replacing missing values by media gruped by product
df['Price'] = df.groupby('Product')['Price'].transform(lambda x: x.fillna(x.mean()))

#Turn negative values into positive, because Price must be positive
df['Price'] = round(df['Price'].abs(), 2)

#Verify if there are missing values
print(df['Price'].isna().sum())

     ID Customer_Name   Order_ID Order_Date        Product  Category  \
16  116  Customer_116  ORD-63660 2025-10-30      Microwave      Home   
24  124  Customer_124  ORD-46136 2025-05-31         Comics     Books   
30  130  Customer_130  ORD-34007 2025-08-15  Tennis racket    Sports   
56  156  Customer_156  ORD-34679 2024-11-28          Jeans  Clothing   
83  183  Customer_183  ORD-20916 2025-03-10          Shoes  Clothing   

    Quantity Price    Payment_Method      Status  Total  
16         4   NaN  Cash on Delivery  Processing    NaN  
24         5   NaN            PayPal   Cancelled    NaN  
30         2   NaN     Bank Transfer  Processing    NaN  
56         2   NaN     Bank Transfer     Shipped    NaN  
83         5   NaN  Cash on Delivery    Returned    NaN  
<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 n

In [232]:
#====CLEANING Payment_method COLUMN======
print(df['Payment_Method'].value_counts())
#There isn't any missing value

#Transform the column into category data type
df['Payment_Method'] = df['Payment_Method'].astype('category')
print(df.info())

Payment_Method
Cash on Delivery    33
PayPal              24
Bank Transfer       22
Credit Card         22
Name: count, dtype: int64
<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 non-null    int64         
 1   Customer_Name   101 non-null    str           
 2   Order_ID        101 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         101 non-null    str           
 5   Category        101 non-null    category      
 6   Quantity        101 non-null    int64         
 7   Price           101 non-null    float64       
 8   Payment_Method  101 non-null    category      
 9   Status          101 non-null    str           
 10  Total           88 non-null     float64       
dtypes: category(2), datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 12.1 KB
None


In [233]:
#======CLEANING status COLUMN======
print(df['Status'].value_counts())
#There is not any problem
#Transform the column into categorical data type
df['Status'] = df['Status'].astype('category')
print(df.info())

Status
Processing    26
Returned      26
Shipped       23
Cancelled     15
Delivered     11
Name: count, dtype: int64
<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 non-null    int64         
 1   Customer_Name   101 non-null    str           
 2   Order_ID        101 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         101 non-null    str           
 5   Category        101 non-null    category      
 6   Quantity        101 non-null    int64         
 7   Price           101 non-null    float64       
 8   Payment_Method  101 non-null    category      
 9   Status          101 non-null    category      
 10  Total           88 non-null     float64       
dtypes: category(3), datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 10.7 KB
None


In [234]:
#======CLEANING Total COLUMN==========
#The total is the Quantity times Price
df['Total'] = round(df['Quantity']*df['Price'], 2)
print(df.info())

<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 non-null    int64         
 1   Customer_Name   101 non-null    str           
 2   Order_ID        101 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         101 non-null    str           
 5   Category        101 non-null    category      
 6   Quantity        101 non-null    int64         
 7   Price           101 non-null    float64       
 8   Payment_Method  101 non-null    category      
 9   Status          101 non-null    category      
 10  Total           101 non-null    float64       
dtypes: category(3), datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 10.7 KB
None


In [235]:
#View the clean data set
print(df.info())
df.head()

#The dataset is clean each column with their respective data type

<class 'pandas.DataFrame'>
Index: 101 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              101 non-null    int64         
 1   Customer_Name   101 non-null    str           
 2   Order_ID        101 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         101 non-null    str           
 5   Category        101 non-null    category      
 6   Quantity        101 non-null    int64         
 7   Price           101 non-null    float64       
 8   Payment_Method  101 non-null    category      
 9   Status          101 non-null    category      
 10  Total           101 non-null    float64       
dtypes: category(3), datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 10.7 KB
None


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,465.42,Cash on Delivery,Shipped,1396.26
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,560.99,PayPal,Processing,1121.98
2,102,Customer_102,ORD-84355,2024-12-23,Tennis racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [236]:
#Export the clean data
df.to_csv('clean_data.csv', index = False)